In [10]:
import numpy as np
import pandas as pd
import random
import torch

from sklearn.model_selection import train_test_split
from torchvision import transforms
from torch.utils.data import Dataset
from PIL import Image

import sys
from pathlib import Path

PROJECT_ROOT = Path("..")
sys.path.append(str(PROJECT_ROOT))

from app.preprocessing import calculate_mean_std, get_cnn_transform

In [11]:
SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

In [12]:
train_path = "../data/fashion-mnist_train.csv"
test_path = "../data/fashion-mnist_test.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

In [13]:
train_data, val_data = train_test_split(
    train_df,
    test_size=0.2,
    random_state=SEED,
    stratify=train_df["label"]
)

print(train_data.shape)
print(val_data.shape)
print(test_df.shape)

(48000, 785)
(12000, 785)
(10000, 785)


In [14]:
# Image preprocessing
fashion_mnist_mean, fashion_mnist_std = calculate_mean_std(train_data)

print(f"Mean: {fashion_mnist_mean:.4f}")
print(f"Std: {fashion_mnist_std:.4f}")

Mean: 0.2857
Std: 0.3525


In [15]:
# uses the original Fashion MNIST format: 1×28×28 grayscale images.
# normalize using one mean and one standard deviation value.

transform_cnn = get_cnn_transform(
    fashion_mnist_mean,
    fashion_mnist_std
)

In [16]:
train_data.to_csv("../data/train_data.csv", index=False)
val_data.to_csv("../data/val_data.csv", index=False)
test_df.to_csv("../data/test_data.csv", index=False)

In [17]:
preprocessing_params = {
    "fashion_mnist_mean": fashion_mnist_mean,
    "fashion_mnist_std": fashion_mnist_std
}

pd.Series(preprocessing_params).to_json("../data/preprocessing_params.json")